# LLM-as-Judge — Standalone Triage Notebook

An **optional add-on** to the NetSec Dashboard. The main dashboard
(`app/Network_Security_Dashboard.ipynb`) runs completely independently of
this notebook — nothing here is required for it, and nothing here changes it.

**What this does.** The detection pipeline already produces several
independent signals per entity: IsolationForest outlier scores, DBSCAN noise
labels, and deterministic rule alerts (port scans, SYN floods, DNS
amplification, ARP spoofing). This notebook fuses them: every flagged
candidate is sent — with *all* of its signals in one compact JSON blob — to
an LLM that returns one strict-JSON verdict (`benign | suspicious |
malicious`), an attack category, a confidence, the evidence used, and a
one-paragraph reasoning trace. The queue you see at the end is re-ranked by
an ensemble of the ML score, the judge's confidence, and the category
severity, and protected by a **rule guardrail**: a candidate whose
deterministic rule fired can never be downgraded to benign by the model.

The judge **never acts**: `recommended_action` is advice for a human, wired
to nothing.

**Prerequisites**

1. Project basics: `pip install -r requirements.txt` + `tshark` on PATH
   (auto-detected in the standard Wireshark install folders).
2. One LLM provider (pick any — all bring-your-own):

| Provider | Setup | Cost |
|---|---|---|
| **Claude API** (default, best quality) | set `ANTHROPIC_API_KEY` + `pip install -r llm_judge/requirements.txt` | your own key, cents per PCAP |
| **Ollama** (local) | install from ollama.com, `ollama pull llama3.2`, set `LLM_JUDGE_PROVIDER=ollama` | free |
| **OpenAI-compatible** (LM Studio local server, or any hosted OpenAI-style endpoint) | set `LLM_JUDGE_PROVIDER=openai_compat`, `OPENAI_COMPAT_BASE_URL`, `OPENAI_COMPAT_MODEL` (+ `OPENAI_COMPAT_API_KEY` if the endpoint needs one) | free locally; hosted per its own terms |

No key is ever stored in this repo; keys are read from the environment at
call time only.

Run the cells top to bottom. Design document: `docs/LLM_JUDGE_SPEC.md`;
usage details: `llm_judge/README.md`.


In [ ]:
# --- Setup: paths, imports, and a readiness check -------------------------
import os, sys, json, shutil

HERE = os.getcwd()
ROOT = HERE if os.path.isdir(os.path.join(HERE, "llm_judge")) \
    else os.path.dirname(HERE)
for _p in (ROOT, os.path.join(ROOT, "attack_tests")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from llm_judge import judge_config, judge_core, calibration, benchmark
from llm_judge.llm_clients import (make_client, ClaudeClient, OllamaClient,
                                   OpenAICompatClient)

print("Provider       :", judge_config.LLM_JUDGE_PROVIDER)
if judge_config.LLM_JUDGE_PROVIDER == "claude":
    print("Model          :", judge_config.CLAUDE_MODEL)
    _key = bool(os.environ.get("ANTHROPIC_API_KEY"))
    print("API key        :", "found in environment" if _key else
          "MISSING - set ANTHROPIC_API_KEY, or switch provider (README)")
elif judge_config.LLM_JUDGE_PROVIDER == "ollama":
    print("Model          :", judge_config.OLLAMA_MODEL,
          "@", judge_config.OLLAMA_HOST)
else:
    print("Model          :", judge_config.OPENAI_COMPAT_MODEL or
          "MISSING - set OPENAI_COMPAT_MODEL",
          "@", judge_config.OPENAI_COMPAT_BASE_URL)
print("Prompt version :", judge_config.PROMPT_VERSION)
print("Rule guardrail :", "on" if judge_config.RULE_GUARDRAIL else "off")
print("Batch cap      :", judge_config.MAX_CANDIDATES_PER_BATCH,
      "candidates")
print("Verdict cache  :", judge_config.CACHE_DB)
print("tshark         :", shutil.which("tshark") or
      "MISSING - install Wireshark (auto-detected when installed)")


## 0 · Benchmark a model before trusting it (recommended)

`llm_judge/benchmark_fixtures.json` holds **11 labeled candidates extracted
from the five real attack PCAPs** by the exact production pipeline. This
section judges all of them with any model you name and scores it — so you
know whether a model is good enough *before* reading its verdicts on your
own captures. No tshark needed; a couple of minutes per local model.

What the numbers mean:

- **detection rate** — attack candidates judged non-benign. With the rule
  guardrail on, this should be 1.0 for any model that returns valid JSON.
- **category accuracy** — exact attack-category match; this is where model
  quality actually shows.
- **benign accuracy** — ML-only outliers correctly left benign (a weak
  model that cries "malicious" at everything fails here).

List the models you want to compare and run:


In [ ]:
# --- Benchmark any models you have ------------------------------------------
import pandas as pd

MODELS_TO_BENCH = [
    # ("ollama", "llama3.2"),
    # ("openai_compat", "your-model-name"),
    # ("claude", "claude-opus-4-8"),
]

_CLIENT_CLS = {"claude": ClaudeClient, "ollama": OllamaClient,
               "openai_compat": OpenAICompatClient}

if MODELS_TO_BENCH:
    reports = []
    for provider, model in MODELS_TO_BENCH:
        print(f"\n===== {provider} / {model} =====")
        bench_client = _CLIENT_CLS[provider](
            model=model, verdict_schema=judge_core.VERDICT_SCHEMA)
        reports.append(benchmark.run_benchmark(bench_client))
    summary = pd.DataFrame([{
        "model": r["model"],
        "category_accuracy": r["category_accuracy"],
        "detection_rate": r["detection_rate"],
        "benign_accuracy": r["benign_accuracy"],
        "guardrail_saves": r["guardrail_saves"],
        "dropped": r["dropped"],
        "median_latency_ms": r["latency_ms_median"],
        "verdict": benchmark.verdict_line(r),
    } for r in reports])
    display(summary)
else:
    print("No models listed - edit MODELS_TO_BENCH above to qualify a "
          "model, or skip ahead if you already trust your configured one.")


## 1 · Analyze a capture

The judge reuses the **exact detection code paths of the dashboard**
(`attack_tests/run_pipeline.py`): tshark feature extraction, the
IsolationForest seed-stability sweep, DBSCAN with auto-eps, and the
deterministic rule layer. Point `PCAP_PATH` at any capture — the five
labeled attack PCAPs under `attack_tests/pcaps/` are a good start:

| PCAP | Attack |
|---|---|
| `tcp_syn_scan.pcap` | nmap SYN scan |
| `xmas_scan.pcap` | nmap Xmas scan |
| `arpspoof.pcap` | ARP-poison MITM |
| `synflood.pcap` | spoofed SYN flood (37k sources) |
| `dns_amp.pcap` | DNS amplification (victim side) |


In [ ]:
# --- Pick a capture and run the detection pipeline ------------------------
PCAP_PATH = os.path.join(ROOT, "attack_tests", "pcaps", "tcp_syn_scan.pcap")

import run_pipeline as rp   # the dashboard's extraction + ML + rules

S = rp.analyze_pcap(PCAP_PATH, "S1")
rp.run_ml_on_session(S)
findings = rp.run_security_scans(S)


## 2 · Assemble the candidate queue

A **candidate** is anything at least one detector flagged: an
IsolationForest majority-vote anomaly, a DBSCAN noise point (only when the
clustering is meaningful — an all-noise result, e.g. under a spoofed flood,
is ignored), or any rule alert. Aggregate floods additionally produce one
**session-level** candidate, because thousands of spoofed sources have no
meaningful per-IP identity.

Each candidate becomes a compact JSON blob (schema:
`llm_judge/schemas/candidate_context.schema.json`) carrying the 10 ML
features, the ML signals, and the rule alerts. A hard batch cap
(default 40) protects against pathological captures — rule-triggered
candidates always survive the cap.


In [ ]:
# --- Build candidate blobs -------------------------------------------------
import pandas as pd

assembled = judge_core.assemble_candidates(S, findings)
candidates = assembled["candidates"]
print(f"{len(candidates)} candidate(s) to judge; "
      f"{len(assembled['capped'])} capped out by the batch limit\n")

preview = pd.DataFrame([{
    "candidate": c["candidate_id"],
    "kind": c["kind"],
    "triggers": ", ".join(c["trigger_reasons"]),
    "iso_score": c["ml_signals"]["iso_score"],
    "scan_alerts": len(c["rule_signals"]["scan_alerts"]),
    "amp_alerts": len(c["rule_signals"]["amp_alerts"]),
    "arp_multi_mac": c["rule_signals"]["arp_multi_mac"],
} for c in candidates])
preview


## 3 · Judge

One LLM call per candidate. The response is **schema-enforced JSON**
(structured outputs on all providers) and is validated again client-side;
an invalid response is retried once, then the candidate is dropped and
reported — a single bad response never poisons the batch.

Two safety nets sit on top of the model:

- **Verdict cache** (SQLite, keyed by candidate blob + prompt version +
  model): re-running the same capture is instant and free.
- **Rule guardrail**: if a deterministic rule fired and the model still
  says benign, the verdict is raised to *suspicious* with the rule-implied
  category. The model's original verdict is kept in the result (and in the
  ⚑ column below) so you always see when the guardrail intervened.

The final queue is ranked by the ensemble priority:
`0.20·anomaly + 0.40·judge confidence + 0.30·category severity`.


In [ ]:
# --- Run the judge ----------------------------------------------------------
client = make_client(verdict_schema=judge_core.VERDICT_SCHEMA)
out = judge_core.judge_candidates(candidates, client=client)

if out["dropped"]:
    print("\nDropped (LLM failure after one retry):")
    for d in out["dropped"]:
        print("  -", d["candidate_id"], "->", d["error"])
print("\nstats:", out["stats"])


In [ ]:
# --- Triaged queue: verdict table ranked by ensemble priority ---------------
rows = pd.DataFrame([{
    "#": i + 1,
    "candidate": r["candidate_id"],
    "verdict": r["verdict"]["verdict"],
    "category": r["verdict"]["category"],
    "confidence": r["verdict"]["confidence"],
    "priority": r["priority"],
    "guardrail": "⚑" if r["guardrail"] else "",
    "action": r["verdict"]["recommended_action"],
    "evidence": ", ".join(r["verdict"]["evidence_features"][:4]),
    "reasoning": r["verdict"]["reasoning"],
} for i, r in enumerate(out["results"])])

_VERDICT_BG = {"malicious": "#7f1d1d", "suspicious": "#92400e",
               "benign": "#14532d"}

def _color_verdicts(col):
    return [f"background-color: {_VERDICT_BG.get(v, '')}; color: white; "
            f"font-weight: bold" for v in col]

if len(rows):
    styled = (rows.style
              .apply(_color_verdicts, subset=["verdict"])
              .format({"confidence": "{:.2f}", "priority": "{:.3f}"})
              .hide(axis="index"))
    display(styled)
else:
    print("No verdicts to show (all candidates dropped, or judge disabled).")


In [ ]:
# --- Persist the batch to llm_judge/output/ as JSON -------------------------
saved_to = judge_core.save_verdicts(out, PCAP_PATH)
print("Verdicts written to:", saved_to)


## 3.5 · Expert panel — several judges that argue (optional)

One model can hallucinate; a panel has to convince itself. With
`LLM_JUDGE_PANEL` set, every candidate is judged **independently by N
models**; when they disagree on the verdict or the category, each judge
receives the peers' anonymized analyses and must either **revise** its
position or **defend** it with a rebuttal grounded in the candidate blob
(one debate round — agreed candidates never cost extra calls). A
deterministic resolver then produces the effective verdict: consensus
takes the highest-confidence verdict; a surviving dispute takes the
fail-safe, more severe side and is flagged ⚖ for human review. The rule
guardrail and the verdict cache apply exactly as in the single-judge path.

The stats carry a **participation report** — per model: candidates
received / valid verdicts / failures / debates / revisions / agreement
with the final verdict / latency — so a silently failing judge is
impossible to miss, and the run keeps going as long as at least one
judge still answers.

In [ ]:
# --- Expert panel (optional): several judges + a debate round ---------------
# Configure via env before starting Jupyter, e.g.
#   LLM_JUDGE_PANEL=llama-3.3-70b-versatile,llama-3.1-8b-instant
# or set PANEL_SPEC here for an ad-hoc run (the env var wins when both set).
PANEL_SPEC = ""   # e.g. "ollama:llama3.2,ollama:gemma3:4b"

spec = judge_config.LLM_JUDGE_PANEL or PANEL_SPEC
if spec:
    from llm_judge.llm_clients import make_panel_clients
    entries = judge_core.parse_panel_spec(spec)
    panel_clients, init_failures = make_panel_clients(
        entries, verdict_schema=judge_core.VERDICT_SCHEMA)
    for f in init_failures:
        print(f"judge {f['entry']} failed to initialize: {f['error']}")
    panel_out = judge_core.judge_candidates_panel(candidates, panel_clients)

    prow = pd.DataFrame([{
        "candidate": r["candidate_id"],
        "verdict": r["verdict"]["verdict"],
        "category": r["verdict"]["category"],
        "debated": "yes" if r["panel"]["debate"] else "",
        "review": "⚖" if r["panel"]["needs_human_review"] else "",
        "judges": " | ".join(
            f"{j['model']}: "
            f"{j['verdict']['verdict'] if j['verdict'] else 'failed'}"
            + (" ↺" if j["revised"] else "")
            for j in r["panel"]["judges"]),
    } for r in panel_out["results"]])
    display(prow)

    print("\nPanel participation report (per-judge audit):")
    display(pd.DataFrame(panel_out["stats"]["panel_report"]).T)
else:
    print("Panel off - set LLM_JUDGE_PANEL (or PANEL_SPEC above) to run "
          "the same candidates through several models with a debate round.")


## 4 · Calibration against labeled ground truth (recommended)

`attack_tests/ground_truth.json` labels five real attack PCAPs
exhaustively. Calibration runs the judge over all five and computes
**Cohen's kappa** between the judge's categories and the labels — the one
number that guards prompt drift:

- per-IP candidates score against the labeled entity lists (an unlisted
  flagged IP counts as `benign_anomaly` — exactly the false positives the
  judge should down-rank);
- aggregate-flood PCAPs score **only** the session candidate, because
  spoofed sources have no per-IP identity.

The report is written to `llm_judge/calibration/results/<version>.json`
and is meant to be **committed** — `tests/test_judge_kappa_regression.py`
gates CI on it without ever calling an LLM. After every prompt edit: bump
`PROMPT_VERSION` in `judge_config.py`, re-run this section, record the
kappa in `PROMPT_CHANGELOG.md`, and commit only if kappa did not regress.

Expect a few minutes: five PCAPs are analyzed (the flood capture is large)
and every uncached candidate is one LLM call.


In [ ]:
# --- Calibration run (flip the switch to execute) ---------------------------
RUN_CALIBRATION = False

if RUN_CALIBRATION:
    report = calibration.run_calibration(client=client)
    overall = report["overall"]
    print(json.dumps({k: v for k, v in overall.items()
                      if k != "confusion"}, indent=2))
    kappa = overall["category_kappa_linear"]
    verdict_gate = (kappa is not None
                    and kappa >= judge_config.KAPPA_THRESHOLD)
    print(f"\ncategory kappa (linear) = {kappa}  ->  "
          f"{'MEETS' if verdict_gate else 'BELOW'} threshold "
          f"{judge_config.KAPPA_THRESHOLD}")
else:
    existing = calibration.latest_calibration_result()
    if existing:
        print("Latest committed calibration report:")
        print("  prompt :", existing["prompt_version"])
        print("  model  :", existing["model"])
        print("  overall:", json.dumps(
            {k: v for k, v in existing["overall"].items()
             if k != "confusion"}, indent=2))
    else:
        print("No calibration report exists yet. Set "
              "RUN_CALIBRATION = True and re-run this cell to produce "
              "the first one.")


## Notes & troubleshooting

| Symptom | Fix |
|---|---|
| `API key MISSING` in the setup cell | Set `ANTHROPIC_API_KEY` in the environment **before** starting Jupyter, or switch provider (see the table at the top) |
| `Ollama call failed ...` | Start the daemon and make sure the model is pulled (`ollama pull <model>`) |
| `OpenAI-compatible call failed ...` | Check `OPENAI_COMPAT_BASE_URL` points at a running server (LM Studio: enable the local server) and `OPENAI_COMPAT_MODEL` names a loaded model |
| `tshark MISSING` | Install Wireshark — the standard install folders are auto-detected, no PATH editing needed |
| Verdicts look stale after editing the prompt | Bump `PROMPT_VERSION` in `judge_config.py` — the cache key includes it. Deleting `llm_judge/cache/` also forces a full re-judge |
| A candidate shows as *dropped* | The provider returned invalid JSON twice (or refused/timed out). The rest of the batch is unaffected; re-run the judge cell to retry |
| Every verdict has ⚑ (guardrail) | The model keeps calling rule-fired attacks benign — it is too weak for this task. Benchmark it (section 0) and pick a stronger one |
| Batch smaller than expected | The per-batch cap (default 40) trimmed statistical-only candidates; raise `LLM_JUDGE_MAX_CANDIDATES` if you want them all |

**Scope boundaries** (by design): the judge only runs here, at PCAP load
time — live-capture streaming is out of scope; verdicts re-rank a queue and
explain themselves, they never block or modify traffic; and the main
dashboard remains fully functional without this folder.
